# MIMII Self-Supervised ASD — DINO Self-Distillation

## Where this notebook sits

The **baseline** notebook learned the normal sound manifold by *reconstruction*: a fully-connected
autoencoder trained on normal clips, with the reconstruction error used as the anomaly score.

This notebook follows a **self-supervised representation learning** route instead, adapting the method of
[**AADSCL** — *Self-Supervised Acoustic Anomaly Detection via Contrastive Learning*](https://github.com/Armanfard-Lab/AADSCL)
(Hojjati & Armanfard). AADSCL trains a ResNet-18 encoder on mel spectrograms with a **contrastive (NT-Xent)**
objective plus an auxiliary transform-classification head, then scores anomalies by the **Mahalanobis distance**
of a clip's embedding from the normal training distribution.

We keep AADSCL's overall recipe — *learn an embedding from normals only, then score with Mahalanobis distance* —
but **replace the contrastive objective with [DINO](https://arxiv.org/abs/2104.14294)
(self-**di**stillation with **no** labels).**

## Why DINO instead of contrastive learning

| | Contrastive (AADSCL) | DINO (this notebook) |
|---|---|---|
| Signal | pull augmentations of the same clip together, push *different* clips apart | match a **student**'s output distribution to a momentum **teacher**'s on different views |
| Negatives | needs negative pairs → sensitive to batch size / false negatives | **no negative pairs** — avoids treating two genuinely-similar normal clips as negatives |
| Collapse control | temperature + many negatives | teacher **centering + sharpening** |
| Views | two augmented views | **multi-crop**: 2 global + several local views |

For anomaly detection where *all* training clips are "normal" and therefore mutually similar, the lack of
negative pairs is appealing: contrastive learning can be hurt by pushing apart normal clips that *should* be
close. DINO sidesteps this entirely.

## The pipeline

```
normal clips ──► log-mel spectrogram ──► multi-crop views ──► student / teacher ResNet-18
                                                                     │
                                              DINO self-distillation loss (train on NORMALS only)
                                                                     │
                          teacher backbone ──► clip embedding ──► Mahalanobis distance = anomaly score
```

Training is unsupervised (normals only). Anomaly labels are used **only** at evaluation time for AUC-ROC,
exactly as in the baseline so the two are directly comparable.

In [1]:
# ── imports ───────────────────────────────────────────────────────────────────
import random, math, copy
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.covariance import LedoitWolf

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

Using device: cuda


## 1 — Configuration

All tunable parameters live here. The spectrogram settings (`N_FFT`, `HOP_LEN`, `N_MELS`) match the baseline
so the feature front-end is identical and only the *learning objective* differs.

DINO-specific knobs:
- **Multi-crop** — `N_GLOBAL` wide views (`G_WIDTH` frames) the teacher sees, plus `N_LOCAL` short views
  (`L_WIDTH` frames) only the student sees. Cropping is along the **time** axis; the full mel-frequency range
  is always kept.
- **Temperatures** — the teacher uses a *low* temperature (`TEACHER_TEMP`) to produce sharp targets; the
  student a *higher* one (`STUDENT_TEMP`). The gap is what drives learning.
- **Momentum** — `TEACHER_MOMENTUM` controls how slowly the teacher tracks the student (EMA).
- **`OUT_DIM`** — dimensionality of the DINO prototype space (the head's output); large by design.

In [2]:
#Using Colab?
COLAB = False

In [4]:
# ── audio ─────────────────────────────────────────────────────────────────────
if COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = Path('drive/MyDrive/ASD-with-SSL/data/fan')
else:
    DATA_ROOT = Path('fan')
MACHINE_IDS = ['id_00', 'id_02', 'id_04', 'id_06']
SR          = 16_000
CHANNEL     = 0

# ── spectrogram (identical to the baseline front-end) ─────────────────────────
N_FFT       = 1024
HOP_LEN     = 512
N_MELS      = 64        # one full mel-spectrogram is (64, ~313) for a 10 s clip

# ── DINO multi-crop ───────────────────────────────────────────────────────────
G_WIDTH     = 128       # global crop width in frames (~4.1 s)  → teacher + student
L_WIDTH     = 44        # local  crop width in frames (~1.4 s)  → student only
N_GLOBAL    = 2
N_LOCAL     = 4
N_CROPS     = N_GLOBAL + N_LOCAL

# ── DINO head / loss ──────────────────────────────────────────────────────────
OUT_DIM         = 4096   # prototype dimensionality
BOTTLENECK_DIM  = 256
HIDDEN_DIM      = 512
TEACHER_TEMP    = 0.04
STUDENT_TEMP    = 0.10
CENTER_MOMENTUM = 0.9
TEACHER_MOMENTUM = 0.996

# ── training ──────────────────────────────────────────────────────────────────
EPOCHS      = 30
BATCH_SIZE  = 64
LR          = 5e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 4
VAL_FRAC    = 0.1

print(f'Global crop: {G_WIDTH} frames | Local crop: {L_WIDTH} frames | '
      f'{N_GLOBAL} global + {N_LOCAL} local views per clip')

Global crop: 128 frames | Local crop: 44 frames | 2 global + 4 local views per clip


## 2 — Data manifest & splits

Identical split logic to the baseline so results are comparable:

- **Train** — 90% of normal files → used to train DINO (no labels)
- **Val** — 10% of normal files → held-out normals, used as the "normal" class at evaluation
- **Test** — all abnormal files → the "anomaly" class at evaluation

Mahalanobis statistics are fit on **train** normals only; AUC is computed over **val + test**.

In [5]:
records = []
for machine_id in MACHINE_IDS:
    for label in ('normal', 'abnormal'):
        for wav in sorted((DATA_ROOT / machine_id / label).glob('*.wav')):
            records.append({'machine_id': machine_id, 'label': label, 'path': wav})

manifest = pd.DataFrame(records)
manifest['is_anomaly'] = (manifest['label'] == 'abnormal').astype(int)

for machine_id in MACHINE_IDS:
    normal   = manifest[(manifest['machine_id'] == machine_id) & (manifest['label'] == 'normal')]
    abnormal = manifest[(manifest['machine_id'] == machine_id) & (manifest['label'] == 'abnormal')]
    train_idx, val_idx = train_test_split(normal.index, test_size=VAL_FRAC, random_state=SEED)
    manifest.loc[train_idx, 'split'] = 'train'
    manifest.loc[val_idx,   'split'] = 'val'
    manifest.loc[abnormal.index, 'split'] = 'test'

print(manifest.groupby(['machine_id', 'split']).size().unstack(fill_value=0)[['train', 'val', 'test']])

split       train  val  test
machine_id                  
id_00         909  102   407
id_02         914  102   359
id_04         929  104   348
id_06         913  102   361


## 3 — Feature front-end & normalisation

Each clip becomes a single **log-mel spectrogram** of shape `(N_MELS, T)` — we do *not* slice it into tiny
frame-vectors as the baseline did, because the ResNet encoder consumes the 2-D spectrogram directly and DINO's
crops provide the windowing.

We standardise spectrograms to zero mean / unit variance using a **single global mean and std** computed over
the training spectrograms (across all machine IDs). This is the image-style normalisation a CNN expects, and
using train-only statistics prevents test-set leakage.

In [6]:
def load_mono(path):
    audio, _ = sf.read(path, dtype='float32')
    return audio[:, CHANNEL] if audio.ndim == 2 else audio

def wav_to_logmel(path):
    """Return a log-mel spectrogram of shape (N_MELS, T)."""
    y = load_mono(path)
    S = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT,
                                        hop_length=HOP_LEN, n_mels=N_MELS)
    return librosa.power_to_db(S, ref=np.max).astype(np.float32)

# ── global normalisation statistics from training spectrograms ────────────────
print('Computing normalisation statistics from training files...')
train_paths_all = manifest[manifest['split'] == 'train']['path'].tolist()

_sum = _sqsum = _count = 0.0
for p in tqdm(train_paths_all):
    s = wav_to_logmel(p)
    _sum   += s.sum()
    _sqsum += (s ** 2).sum()
    _count += s.size

NORM_MEAN = float(_sum / _count)
NORM_STD  = float(np.sqrt(_sqsum / _count - NORM_MEAN ** 2)) + 1e-8
print(f'Global mel mean={NORM_MEAN:.3f}  std={NORM_STD:.3f}')

Computing normalisation statistics from training files...


  0%|          | 0/3665 [00:00<?, ?it/s]

Global mel mean=-25.691  std=9.298


## 4 — Multi-crop views & augmentation

DINO's core data trick is **multi-crop**. From each spectrogram we draw:

- `N_GLOBAL` **global crops** — wide time windows (`G_WIDTH` frames) that capture long-range structure,
- `N_LOCAL` **local crops** — short windows (`L_WIDTH` frames) that capture fine detail.

The teacher only ever sees global crops; the student sees *all* of them. Asking the student's prediction on a
small local patch to match the teacher's prediction on the whole context is what produces useful "local-to-global"
representations.

On top of cropping we apply light, audio-appropriate augmentation to every crop — **SpecAugment** (random
frequency-band and time masking), additive Gaussian noise, and a small random gain. These are the spectrogram
analogue of AADSCL's audio transforms, and they are what make the two views of a clip genuinely *different*
so the self-distillation target is non-trivial.

In [7]:
def spec_augment(w, max_freq_mask=12, time_mask_frac=0.25,
                 noise_std=0.05, gain_range=(0.9, 1.1)):
    """Light SpecAugment-style augmentation on a single (N_MELS, width) crop."""
    w = w.copy()
    # frequency mask
    f  = random.randint(0, max_freq_mask)
    f0 = random.randint(0, max(0, w.shape[0] - f))
    w[f0:f0 + f, :] = 0.0
    # time mask
    max_t = max(1, int(w.shape[1] * time_mask_frac))
    t  = random.randint(0, max_t)
    t0 = random.randint(0, max(0, w.shape[1] - t))
    w[:, t0:t0 + t] = 0.0
    # additive noise + random gain
    w = w + np.random.randn(*w.shape).astype(np.float32) * noise_std
    w = w * np.float32(random.uniform(*gain_range))
    return w

def make_crop(spec, width):
    """Random time-crop of given width (pad if the clip is shorter), then augment."""
    T = spec.shape[1]
    if T <= width:
        w = np.pad(spec, ((0, 0), (0, width - T)))
    else:
        s = random.randint(0, T - width)
        w = spec[:, s:s + width]
    w = spec_augment(w)
    return torch.from_numpy(w[None, :, :].copy())   # (1, N_MELS, width)

class MelCropDataset(Dataset):
    """Returns a list of multi-crop views for one clip: global crops first, then locals.

    The default PyTorch collate turns a batch of these lists into a list of stacked
    tensors — one [B, 1, N_MELS, width] tensor per crop position — which is exactly
    what the multi-crop forward pass expects.
    """
    def __init__(self, paths, mean, std):
        self.paths = list(paths); self.mean = mean; self.std = std

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        spec = (wav_to_logmel(self.paths[idx]) - self.mean) / self.std
        crops  = [make_crop(spec, G_WIDTH) for _ in range(N_GLOBAL)]
        crops += [make_crop(spec, L_WIDTH) for _ in range(N_LOCAL)]
        return crops

## 5 — Model: ResNet-18 backbone + DINO projection head

AADSCL uses a ResNet-18 encoder on 1-channel mel spectrograms; we keep that backbone but implement it in pure
PyTorch (no `torchvision` dependency). A global average pool makes the backbone accept crops of *any* width, so
global and local crops flow through the same network unchanged.

On top sits the standard **DINO head**: a small MLP → L2-normalised bottleneck → a weight-normalised linear layer
that produces logits over `OUT_DIM` "prototypes". We realise the weight-normalised last layer by L2-normalising
both the features and the prototype weights (equivalent to DINO's frozen-norm setup), which makes the logits
cosine similarities to learned prototypes.

In [8]:
class BasicBlock(nn.Module):
    """Standard ResNet basic block (two 3x3 convs + identity shortcut)."""
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride, 1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, 1, 1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_c)
        self.down  = None
        if stride != 1 or in_c != out_c:
            self.down = nn.Sequential(nn.Conv2d(in_c, out_c, 1, stride, bias=False),
                                      nn.BatchNorm2d(out_c))
    def forward(self, x):
        idt = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.down is not None:
            idt = self.down(x)
        return F.relu(out + idt)

class ResNetEncoder(nn.Module):
    """ResNet-18 for 1-channel spectrograms; adaptive pool → 512-d embedding."""
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(1, 64, 7, 2, 3, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, 2, 1))
        self.layer1 = self._make_layer(64,  64,  2, 1)
        self.layer2 = self._make_layer(64,  128, 2, 2)
        self.layer3 = self._make_layer(128, 256, 2, 2)
        self.layer4 = self._make_layer(256, 512, 2, 2)
        self.pool   = nn.AdaptiveAvgPool2d(1)
        self.embed_dim = 512
    def _make_layer(self, in_c, out_c, blocks, stride):
        layers = [BasicBlock(in_c, out_c, stride)]
        layers += [BasicBlock(out_c, out_c) for _ in range(1, blocks)]
        return nn.Sequential(*layers)
    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x); x = self.layer3(x); x = self.layer4(x)
        return self.pool(x).flatten(1)          # (B, 512)

class DINOHead(nn.Module):
    def __init__(self, in_dim, out_dim=OUT_DIM, hidden=HIDDEN_DIM, bottleneck=BOTTLENECK_DIM):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU(),
            nn.Linear(hidden, bottleneck))
        self.last = nn.Linear(bottleneck, out_dim, bias=False)
    def forward(self, x):
        x = self.mlp(x)
        x = F.normalize(x, dim=-1)                      # L2-normalised bottleneck
        return F.linear(x, F.normalize(self.last.weight, dim=-1))   # cosine logits

class MultiCropWrapper(nn.Module):
    """Run the backbone once per crop-resolution group, then the shared head."""
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone = backbone
        self.head     = head
    def forward(self, crops):
        if not isinstance(crops, list):
            crops = [crops]
        widths = torch.tensor([c.shape[-1] for c in crops])
        bounds = torch.cumsum(torch.unique_consecutive(widths, return_counts=True)[1], 0)
        start, out = 0, None
        for end in bounds:
            o = self.backbone(torch.cat(crops[start:end]))
            out = o if out is None else torch.cat((out, o))
            start = end
        return self.head(out)

def build_dino():
    return MultiCropWrapper(ResNetEncoder(), DINOHead(512)).to(DEVICE)

_m = build_dino()
print(f'Trainable parameters: {sum(p.numel() for p in _m.parameters()):,}')
del _m

Trainable parameters: 12,875,456


## 6 — The DINO loss

DINO's objective is a **cross-entropy between the teacher's and the student's softmax distributions** over the
prototypes. Two mechanisms prevent the trivial collapse to a constant output:

1. **Sharpening** — the teacher divides by a small temperature (`TEACHER_TEMP`), producing peaked targets; the
   student uses a larger temperature.
2. **Centering** — a running mean of teacher outputs is subtracted before the teacher softmax, stopping any one
   prototype from dominating.

The loss is summed over all (teacher-global-view, student-view) pairs **except** where the teacher and student
look at the *same* crop, then averaged.

In [9]:
class DINOLoss(nn.Module):
    def __init__(self, out_dim, n_crops, teacher_temp=TEACHER_TEMP,
                 student_temp=STUDENT_TEMP, center_momentum=CENTER_MOMENTUM):
        super().__init__()
        self.student_temp = student_temp
        self.teacher_temp = teacher_temp
        self.center_momentum = center_momentum
        self.n_crops = n_crops
        self.register_buffer('center', torch.zeros(1, out_dim))

    def forward(self, student_out, teacher_out, n_global=N_GLOBAL):
        student = (student_out / self.student_temp).chunk(self.n_crops)
        teacher = F.softmax((teacher_out - self.center) / self.teacher_temp, dim=-1)
        teacher = teacher.detach().chunk(n_global)

        total, n_terms = 0.0, 0
        for ti, t in enumerate(teacher):
            for si, s in enumerate(student):
                if ti == si:        # skip identical global view (teacher vs student on same crop)
                    continue
                total = total + torch.sum(-t * F.log_softmax(s, dim=-1), dim=-1).mean()
                n_terms += 1
        self._update_center(teacher_out)
        return total / n_terms

    @torch.no_grad()
    def _update_center(self, teacher_out):
        batch_center = teacher_out.mean(dim=0, keepdim=True)
        self.center = self.center * self.center_momentum + batch_center * (1 - self.center_momentum)

## 7 — Training loop (one DINO model per machine ID)

The teacher is **not** trained by gradients — after each student update it is nudged toward the student with an
exponential moving average (EMA). We follow the original DINO recipe:

- student updated by AdamW on the DINO loss,
- teacher params ← `momentum · teacher + (1 − momentum) · student`,
- teacher centering buffer updated inside the loss.

As in the baseline we train **one model per machine ID** (each fan unit has its own normal sound signature) and
keep every trained teacher in a dict for the scoring stage.

In [10]:
def train_dino(machine_id):
    train_paths = manifest[(manifest['machine_id'] == machine_id) &
                           (manifest['split'] == 'train')]['path'].tolist()
    loader = DataLoader(MelCropDataset(train_paths, NORM_MEAN, NORM_STD),
                        batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                        pin_memory=True, drop_last=True)

    student = build_dino()
    teacher = build_dino()
    teacher.load_state_dict(student.state_dict())
    for p in teacher.parameters():
        p.requires_grad = False

    loss_fn   = DINOLoss(OUT_DIM, N_CROPS).to(DEVICE)
    optimiser = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    losses = []
    for epoch in range(1, EPOCHS + 1):
        student.train(); teacher.train()
        epoch_loss, n = 0.0, 0
        for crops in loader:
            crops = [c.to(DEVICE, non_blocking=True) for c in crops]
            teacher_out = teacher(crops[:N_GLOBAL])     # teacher sees only global crops
            student_out = student(crops)                # student sees all crops
            loss = loss_fn(student_out, teacher_out, N_GLOBAL)

            optimiser.zero_grad()
            loss.backward()
            optimiser.step()

            with torch.no_grad():                       # EMA teacher update
                for ps, pt in zip(student.parameters(), teacher.parameters()):
                    pt.data.mul_(TEACHER_MOMENTUM).add_(ps.detach().data * (1 - TEACHER_MOMENTUM))

            epoch_loss += loss.item() * crops[0].size(0); n += crops[0].size(0)
        losses.append(epoch_loss / n)
        if epoch % 5 == 0 or epoch == 1:
            print(f'  epoch {epoch:3d}/{EPOCHS}  dino_loss={losses[-1]:.4f}')

    teacher.eval()
    return teacher, losses


teachers, histories = {}, {}
for machine_id in MACHINE_IDS:
    print(f'\n── Training DINO for {machine_id} ──')
    t, ls = train_dino(machine_id)
    teachers[machine_id]  = t
    histories[machine_id] = ls


── Training DINO for id_00 ──
  epoch   1/30  dino_loss=8.0008
  epoch   5/30  dino_loss=7.9953
  epoch  10/30  dino_loss=7.3064
  epoch  15/30  dino_loss=5.7112
  epoch  20/30  dino_loss=4.3369
  epoch  25/30  dino_loss=3.4770
  epoch  30/30  dino_loss=2.8882

── Training DINO for id_02 ──
  epoch   1/30  dino_loss=8.0122
  epoch   5/30  dino_loss=8.0440
  epoch  10/30  dino_loss=7.4329
  epoch  15/30  dino_loss=6.2627
  epoch  20/30  dino_loss=4.3157
  epoch  25/30  dino_loss=2.8778
  epoch  30/30  dino_loss=2.1681

── Training DINO for id_04 ──
  epoch   1/30  dino_loss=8.0086


KeyboardInterrupt: 

### 7.1 — Loss curves

The DINO loss should fall steadily and flatten. Because the teacher is an EMA of the student, the loss does not
go to zero — it settles at the entropy of the (sharpened) matched distributions. A curve that drops then plateaus
without spiking back up indicates stable training (no collapse).

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
fig.suptitle('DINO self-distillation loss per Machine ID', fontsize=12)
for ax, machine_id in zip(axes, MACHINE_IDS):
    ax.plot(histories[machine_id], color='steelblue')
    ax.set_title(machine_id); ax.set_xlabel('Epoch')
    if ax is axes[0]:
        ax.set_ylabel('DINO loss')
plt.tight_layout(); plt.show()

## 8 — Embeddings & Mahalanobis anomaly score

Following AADSCL, the anomaly score is the **Mahalanobis distance** of a clip's embedding from the distribution
of *normal* training embeddings.

We embed a clip with the **teacher backbone** (the representation network, *not* the DINO head — the head is
only a training scaffold). To get a stable, augmentation-free embedding we slide global-width windows across the
clip, embed each, average them, and L2-normalise. Then per machine ID:

1. fit a covariance model (Ledoit-Wolf shrinkage, robust in 512-d) on **train-normal** embeddings,
2. score every **val + test** clip by its (squared) Mahalanobis distance.

Higher distance ⇒ further from the normal manifold ⇒ more anomalous.

In [ ]:
@torch.no_grad()
def embed_clip(path, backbone, width=G_WIDTH):
    """Augmentation-free clip embedding: average backbone features over global windows."""
    spec = (wav_to_logmel(path) - NORM_MEAN) / NORM_STD
    T = spec.shape[1]
    starts = list(range(0, max(1, T - width + 1), width // 2)) or [0]
    windows = []
    for s in starts:
        w = spec[:, s:s + width]
        if w.shape[1] < width:
            w = np.pad(w, ((0, 0), (0, width - w.shape[1])))
        windows.append(w)
    x = torch.from_numpy(np.stack(windows)[:, None, :, :]).to(DEVICE)   # (n, 1, N_MELS, width)
    emb = backbone(x).mean(0).cpu().numpy()
    return emb / (np.linalg.norm(emb) + 1e-8)


results = []
for machine_id in MACHINE_IDS:
    backbone = teachers[machine_id].backbone

    train_paths = manifest[(manifest['machine_id'] == machine_id) &
                           (manifest['split'] == 'train')]['path'].tolist()
    eval_df = manifest[(manifest['machine_id'] == machine_id) &
                       (manifest['split'].isin(['val', 'test']))]

    train_emb = np.stack([embed_clip(p, backbone) for p in tqdm(train_paths, desc=f'{machine_id} fit')])
    eval_emb  = np.stack([embed_clip(p, backbone) for p in eval_df['path']])

    cov = LedoitWolf().fit(train_emb)
    scores = cov.mahalanobis(eval_emb)        # squared Mahalanobis distance

    for (_, row), sc in zip(eval_df.iterrows(), scores):
        results.append({'machine_id': machine_id, 'label': row['label'],
                        'is_anomaly': row['is_anomaly'], 'score': float(sc)})

results_df = pd.DataFrame(results)
print('Scored', len(results_df), 'clips.')

## 9 — Evaluation

### 9.1 — AUC-ROC per machine ID

Same metric and protocol as the baseline (val-normals vs. all abnormals), so the numbers are directly comparable.
The baseline autoencoder reached a mean AUC ≈ **0.61** on this −6 dB fan subset.

In [ ]:
auc_scores = {}
for machine_id in MACHINE_IDS:
    sub = results_df[results_df['machine_id'] == machine_id]
    auc = roc_auc_score(sub['is_anomaly'], sub['score'])
    auc_scores[machine_id] = auc
    print(f'{machine_id}  AUC-ROC: {auc:.4f}')
mean_auc = float(np.mean(list(auc_scores.values())))
print(f'\nMean AUC-ROC: {mean_auc:.4f}')

### 9.2 — ROC curves

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5), sharey=True)
fig.suptitle('ROC Curves per Machine ID (DINO)', fontsize=12)
for ax, machine_id in zip(axes, MACHINE_IDS):
    sub = results_df[results_df['machine_id'] == machine_id]
    fpr, tpr, _ = roc_curve(sub['is_anomaly'], sub['score'])
    ax.plot(fpr, tpr, color='steelblue', linewidth=2)
    ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8)
    ax.set_title(f"{machine_id}\nAUC = {auc_scores[machine_id]:.3f}")
    ax.set_xlabel('False Positive Rate')
    if ax is axes[0]:
        ax.set_ylabel('True Positive Rate')
plt.tight_layout(); plt.show()

### 9.3 — Score distributions

Well-separated normal (blue) vs. abnormal (red) Mahalanobis-distance distributions correspond to high AUC.
Heavy overlap means the learned embedding does not place anomalies far from the normal manifold for that machine.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
fig.suptitle('Mahalanobis-distance distributions per Machine ID', fontsize=12)
colors = {'normal': '#5599dd', 'abnormal': '#dd5555'}
for ax, machine_id in zip(axes, MACHINE_IDS):
    sub = results_df[results_df['machine_id'] == machine_id]
    for label, grp in sub.groupby('label'):
        ax.hist(grp['score'], bins=40, alpha=0.6, color=colors[label], label=label, density=True)
    ax.set_title(machine_id); ax.set_xlabel('Mahalanobis distance')
    if ax is axes[0]:
        ax.set_ylabel('Density'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 10 — Summary & discussion

### Results table

In [ ]:
summary = pd.DataFrame({
    'Machine ID':    MACHINE_IDS,
    'AUC-ROC (DINO)':[round(auc_scores[m], 4) for m in MACHINE_IDS],
    'Train files':   [len(manifest[(manifest['machine_id']==m) & (manifest['split']=='train')]) for m in MACHINE_IDS],
    'Test normal':   [len(manifest[(manifest['machine_id']==m) & (manifest['split']=='val')])   for m in MACHINE_IDS],
    'Test abnormal': [len(manifest[(manifest['machine_id']==m) & (manifest['split']=='test')]) for m in MACHINE_IDS],
}).set_index('Machine ID')
summary.loc['Mean'] = summary.mean()
summary

### What changed vs. the baseline & AADSCL

- **vs. the autoencoder baseline** — we no longer score by reconstruction error. DINO learns an embedding in
  which normal clips form a tight cluster; anomalies are detected by *distance from that cluster*, which can
  capture subtler deviations than per-frame reconstruction.
- **vs. AADSCL** — the front-end (mel spectrogram → ResNet-18) and the back-end (Mahalanobis scoring) are kept,
  but the contrastive NT-Xent loss + transform-classification head are replaced by DINO self-distillation with
  multi-crop. No negative pairs are needed, which suits a training set that is entirely "normal".

### Knobs that matter most

| Lever | Effect |
|---|---|
| `EPOCHS`, `TEACHER_MOMENTUM` | longer training + higher momentum → smoother teacher, usually better features |
| `G_WIDTH` / `L_WIDTH` / `N_LOCAL` | more & smaller local crops → richer local-to-global signal (more compute) |
| `TEACHER_TEMP` | too high → collapse risk; too low → unstable targets. 0.04 is the DINO default |
| augmentation strength (`spec_augment`) | the only source of view difference between the 2 global crops — too weak makes the task trivial |
| scoring rule | swap Ledoit-Wolf Mahalanobis for **k-NN cosine distance** to the train embeddings as a non-parametric alternative |

### Possible next steps

- Pre-extract spectrograms / crops to disk to speed up the data loader.
- Add teacher-temperature warm-up and a cosine LR schedule (the full DINO recipe).
- Average all 8 microphone channels before the mel transform (beamforming) instead of using channel 0.
- Try a k-NN anomaly score and compare against Mahalanobis.